In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
METADATA_PATH = PROJECT_ROOT / "data" / "fma_metadata" / "tracks.csv"

df = pd.read_csv(METADATA_PATH, header=[0, 1], low_memory=False)

print("Shape:", df.shape)

In [ ]:
columns = pd.DataFrame(df.columns.tolist(), columns=["level_1", "level_2"])

columns

In [ ]:
Subset_distribution = (df.set.subset.value_counts())

Subset_distribution

In [ ]:
genre_distribution = df.track.genre_top.value_counts(dropna=False)

genre_distribution

In [ ]:
small = df[df.set.subset == "small"]

print("Number of tracks:", len(small))

split_distribution = small.set.split.value_counts()

split_distribution

In [ ]:
missing_values = small.isnull().sum()

missing_values

In [ ]:
fma_data = small[["track", "set"]].copy()
fma_data = fma_data[fma_data.track.genre_top.notna()].copy()
fma_data = fma_data[fma_data.track.genre_top.isin(fma_data.track.genre_top.value_counts().index)].copy()

print("usable tracks:", len(fma_data))
print("genres:", fma_data.track.genre_top.nunique())
print("\nGenre distribution:")
print(fma_data.track.genre_top.value_counts())

In [ ]:
train_data = fma_data[fma_data["set"]["split"] == "training"].copy()
val_data = fma_data[fma_data["set"]["split"] == "validation"].copy()
test_data = fma_data[fma_data["set"]["split"] == "test"].copy()

print("train:", len(train_data))
print("validation:", len(val_data))
print("test:", len(test_data))

In [ ]:
genres = sorted(train_data["track"]["genre_top"].unique())
genre_to_id = {genre: i for i, genre in enumerate(genres)}
id_to_genre = {i: genre for genre, i in genre_to_id.items()}

train_data["label"] = train_data["track"]["genre_top"].map(genre_to_id)
val_data["label"] = val_data["track"]["genre_top"].map(genre_to_id)
test_data["label"] = test_data["track"]["genre_top"].map(genre_to_id)

genre_to_id
# print("train labels:", train_data["label"].shape)
# print("validation labels:", val_data["label"].shape)
# print("test labels:", test_data["label"].shape)

In [ ]:
print("Train:")
print(train_data["label"].value_counts().sort_index())

print("\nValidation:")
print(val_data["label"].value_counts().sort_index())

print("\nTest:")
print(test_data["label"].value_counts().sort_index())

In [ ]:
import os

AUDIO_DIR = r"G:\gnn-bert-music-context\data\fma_small"

print("Audio directory exists:", os.path.exists(AUDIO_DIR))

if os.path.exists(AUDIO_DIR):
    audio_files = []
    for root, dirs, files in os.walk(AUDIO_DIR):
        for file in files:
            if file.lower().endswith(".mp3"):
                audio_files.append(os.path.join(root, file))

    print("Number of MP3 files:", len(audio_files))

    print("First 5 files:")
    for path in audio_files[:5]:
        print(path)

    print("Last 5 files:")
    for path in audio_files[-5:]:
        print(path)

In [ ]:
from pathlib import Path

track_id_map = small["Unnamed: 0_level_0"]["Unnamed: 0_level_1"].astype(int)

audio_path_map = {
    int(Path(path).stem): path
    for path in audio_files
}

train_data["track_id"] = train_data.index.map(track_id_map)
val_data["track_id"] = val_data.index.map(track_id_map)
test_data["track_id"] = test_data.index.map(track_id_map)

train_data["audio_path"] = train_data["track_id"].map(audio_path_map)
val_data["audio_path"] = val_data["track_id"].map(audio_path_map)
test_data["audio_path"] = test_data["track_id"].map(audio_path_map)

print("Train missing audio:", train_data["audio_path"].isna().sum())
print("Validation missing audio:", val_data["audio_path"].isna().sum())
print("Test missing audio:", test_data["audio_path"].isna().sum())

print("\nExample:")
print(train_data[["track_id", "audio_path", "label"]].head())

In [ ]:
import librosa

sample_path = train_data["audio_path"].iloc[0]

y, sr = librosa.load(sample_path, sr=None, mono=True)

print("Audio path:", sample_path)
print("Sample rate:", sr)
print("Samples:", len(y))
print("Duration:", len(y) / sr, "seconds")
print("Shape:", y.shape)

In [ ]:
import numpy as np
import librosa

def extract_audio_features(audio_path):
    y, sr = librosa.load(audio_path, sr=22050, mono=True)

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
    zero_crossing = librosa.feature.zero_crossing_rate(y)
    rms = librosa.feature.rms(y=y)

    features = np.concatenate([
        mfcc.mean(axis=1),
        mfcc.std(axis=1),
        chroma.mean(axis=1),
        chroma.std(axis=1),
        spectral_centroid.mean(axis=1),
        spectral_bandwidth.mean(axis=1),
        spectral_rolloff.mean(axis=1),
        zero_crossing.mean(axis=1),
        rms.mean(axis=1)
    ])

    return features.astype(np.float32)

sample_features = extract_audio_features(
    train_data["audio_path"].iloc[0]
)

print("Feature shape:", sample_features.shape)
print("First 10 features:", sample_features[:10])

In [ ]:
from tqdm.auto import tqdm

def extract_dataset_features(data):
    features = []
    valid_indices = []

    for idx, path in tqdm(
        data["audio_path"].items(),
        total=len(data),
        desc="Extracting audio features"
    ):
        try:
            feature = extract_audio_features(path)
            features.append(feature)
            valid_indices.append(idx)
        except Exception as e:
            print(f"Error processing track {idx}: {e}")

    return np.array(features, dtype=np.float32), valid_indices

X_train_audio, train_indices = extract_dataset_features(train_data)
X_val_audio, val_indices = extract_dataset_features(val_data)
X_test_audio, test_indices = extract_dataset_features(test_data)

print("Train audio features:", X_train_audio.shape)
print("Validation audio features:", X_val_audio.shape)
print("Test audio features:", X_test_audio.shape)

In [ ]:
y_train = train_data.loc[train_indices, "label"].to_numpy()
y_val = val_data.loc[val_indices, "label"].to_numpy()
y_test = test_data.loc[test_indices, "label"].to_numpy()

print("X_train:", X_train_audio.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val_audio.shape)
print("y_val:", y_val.shape)
print("X_test:", X_test_audio.shape)
print("y_test:", y_test.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_audio = scaler.fit_transform(X_train_audio)
X_val_audio = scaler.transform(X_val_audio)
X_test_audio = scaler.transform(X_test_audio)

print("Train mean:", X_train_audio.mean())
print("Train std:", X_train_audio.std())
print("Validation shape:", X_val_audio.shape)
print("Test shape:", X_test_audio.shape)

In [ ]:
import torch

X_train_tensor = torch.tensor(X_train_audio, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_val_tensor = torch.tensor(X_val_audio, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)

X_test_tensor = torch.tensor(X_test_audio, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

print("X_train:", X_train_tensor.shape)
print("y_train:", y_train_tensor.shape)
print("X_val:", X_val_tensor.shape)
print("y_val:", y_val_tensor.shape)
print("X_test:", X_test_tensor.shape)
print("y_test:", y_test_tensor.shape)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
import torch.nn as nn

class AudioGenreClassifier(nn.Module):
    def __init__(self, input_dim=69, num_classes=8):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.network(x)

model = AudioGenreClassifier(input_dim=69, num_classes=8)

model

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

print("optimizer ready")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

device

In [ ]:
from sklearn.metrics import accuracy_score, f1_score


epochs = 100
patience = 10
best_val_macro_f1 = -1
best_epoch = 0
epochs_without_improvement = 0
train_losses = []
val_losses = []
val_accuracies = []
val_macro_f1s = []
val_micro_f1s = []

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for batch_idx, (X_batch, y_batch) in enumerate(train_loader):
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if batch_idx % 50 == 0:
            print(f"epoch: {epoch + 1} batch: {batch_idx} loss: {loss.item():.4f}")

    train_loss = running_loss / len(train_loader)
    train_losses.append(train_loss)

    model.eval()
    val_loss = 0.0
    val_predictions = []
    val_targets = []

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            val_loss += loss.item()

            predictions = torch.argmax(outputs, dim=1)

            val_predictions.extend(predictions.cpu().numpy())
            val_targets.extend(y_batch.cpu().numpy())

    val_loss /= len(val_loader)

    val_accuracy = accuracy_score(val_targets, val_predictions)
    val_macro_f1 = f1_score(val_targets, val_predictions, average="macro")
    val_micro_f1 = f1_score(val_targets, val_predictions, average="micro")

    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)
    val_macro_f1s.append(val_macro_f1)
    val_micro_f1s.append(val_micro_f1)

    print(f"epoch: {epoch + 1} train loss: {train_loss:.4f} validation loss: {val_loss:.4f}")
    print(f"accuracy: {val_accuracy:.4f} macro-f1: {val_macro_f1:.4f} micro-f1: {val_micro_f1:.4f}")

    if val_macro_f1 > best_val_macro_f1:
        best_val_macro_f1 = val_macro_f1
        best_epoch = epoch + 1
        epochs_without_improvement = 0

        torch.save(model.state_dict(), "best_fma_genre_model.pt")
        print("best model saved")
    else:
        epochs_without_improvement += 1
        print(f"no improvement: {epochs_without_improvement}")

    if epochs_without_improvement >= patience:
        print("early stopping")
        break

print("Best epoch:", best_epoch)
print("Best validation Macro-F1:", best_val_macro_f1)